In [34]:
from convertiq_py.ml_logic.data import load_data_kaggle_raw, clean_data
from convertiq_py.ml_logic.preprocessor import preprocess_features, feature_engineering
from convertiq_py.ml_logic.model import train_test_split, initialize_model, train_model, base_spw
import pandas as pd
from sklearn.model_selection import train_test_split
import lightgbm as lgb

In [35]:
df = load_data_kaggle_raw(10000000)

In [36]:
df.head()

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-10-01 00:00:00 UTC,view,44600062,2103807459595387724,NaN,shiseido,35.790001,541312140,72d76fde-8bb3-4e00-8c23-a032dfed738c
1,2019-10-01 00:00:00 UTC,view,3900821,2053013552326770905,appliances.environment.water_heater,aqua,33.200001,554748717,9333dfbd-b87a-4708-9857-6336556b0fcc
2,2019-10-01 00:00:01 UTC,view,17200506,2053013559792632471,furniture.living_room.sofa,NaN,543.099976,519107250,566511c2-e2e3-422b-b695-cf8e6e792ca8
3,2019-10-01 00:00:01 UTC,view,1307067,2053013558920217191,computers.notebook,lenovo,251.740005,550050854,7c90fc70-0e80-4590-96f3-13c02c18c713
4,2019-10-01 00:00:04 UTC,view,1004237,2053013555631882655,electronics.smartphone,apple,1081.979980,535871217,c6bd7419-2748-4c56-95b4-8cec9ff8b80d


In [38]:
df.shape

(10000000, 9)

In [42]:
df.index

RangeIndex(start=0, stop=10000000, step=1)

In [49]:
dataset = df.iloc[0:100, :]

In [50]:
dataset.to_csv('dataset.csv')

In [ ]:
X_processed, y_processed = preprocess_features(df)

/home/turpindominique/code/AdriMottainai/convertiq/convertiq_py/ml_logic/data.py:55: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["has_valid_price"] = (df["price"] > 0).astype("int8")


✅ Data cleaned
int64
✅ X_processed, with shape (374501, 15)
✅ y_processed, with shape (374501,)


In [3]:
X_train, X_test, y_train, y_test = train_test_split(X_processed, y_processed, test_size=0.2, random_state=42)

In [4]:
X_train.shape

(299600, 15)

In [5]:
y_train.shape

(299600,)

In [6]:
pos = y_train.sum()
float(pos)

7491.0

In [7]:
base_spw(y_train)

39.99466025897744

In [8]:
float((y_train == 0).sum())

292109.0

In [9]:
y_processed.info()

<class 'pandas.core.series.Series'>
RangeIndex: 374501 entries, 0 to 374500
Series name: label
Non-Null Count   Dtype
--------------   -----
374501 non-null  int64
dtypes: int64(1)
memory usage: 2.9 MB


In [10]:
y_processed.value_counts()

label
0    365115
1      9386
Name: count, dtype: int64

In [11]:
test = clean_data(load_data_kaggle_raw(1000000))

✅ Data cleaned


/home/turpindominique/code/AdriMottainai/convertiq/convertiq_py/ml_logic/data.py:55: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["has_valid_price"] = (df["price"] > 0).astype("int8")


In [12]:
test.value_counts()

event_time           event_type  product_id  category_id          category_code           brand      price        user_id    user_session                          has_brand  has_valid_price
2019-10-01 16:56:02  view        21402314    2053013561579406073  electronics.clocks      lorus      104.250000   542975139  618f9790-2a03-4a9f-920a-f1122a53e0f9  1          1                  1
                                 21402762    2053013561579406073  electronics.clocks      unknown    36.290001    555716606  92c82d02-00ac-4053-bb11-855b05007aa1  0          1                  1
2019-10-01 16:56:03  cart        1004870     2053013555631882655  electronics.smartphone  samsung    286.839996   555705674  6b1b6291-61ed-4423-904a-1ec4e53b2677  1          1                  1
                     view        1003855     2053013555631882655  electronics.smartphone  xiaomi     140.910004   514361758  a8a73756-557a-4ecb-bb58-041721a743c0  1          1                  1
                              

In [13]:
X = test

In [14]:
# Observation and prediction set split
observation_end = pd.Timestamp('2019-10-06')
prediction_end  = pd.Timestamp('2019-10-07')

# X_pred sert juste à aller récupérer les acheteurs (y) pendant cette période
X_obs = X[X["event_time"] < observation_end].copy()
X_pred = X[X["event_time"] >= observation_end].copy()

# Create y with purchasers from prediction period
purchasers = set(X_pred.loc[X_pred["event_type"] == "purchase", "user_id"])
y_purchasers = pd.DataFrame({"user_id": X_obs["user_id"].unique()})
y_purchasers["label"] = y_purchasers["user_id"].isin(purchasers).astype(int)

In [15]:
X_pred.head()

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session,has_brand,has_valid_price


In [16]:
X_obs.head(-10)

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session,has_brand,has_valid_price
0,2019-10-01 00:00:01,view,1307067,2053013558920217191,computers.notebook,lenovo,251.740005,550050854,7c90fc70-0e80-4590-96f3-13c02c18c713,1,1
1,2019-10-01 00:00:04,view,1004237,2053013555631882655,electronics.smartphone,apple,1081.979980,535871217,c6bd7419-2748-4c56-95b4-8cec9ff8b80d,1,1
2,2019-10-01 00:00:11,view,1004545,2053013555631882655,electronics.smartphone,huawei,566.010010,537918940,406c46ed-90a4-4787-a43b-59a410c1a5fb,1,1
3,2019-10-01 00:00:11,view,1005011,2053013555631882655,electronics.smartphone,samsung,900.640015,530282093,50a293fb-5940-41b2-baf3-17af0e812101,1,1
4,2019-10-01 00:00:18,view,1801995,2053013554415534427,electronics.video.tv,haier,193.029999,537192226,e3151795-c355-4efa-acf6-e1fe1bebeee5,1,1
...,...,...,...,...,...,...,...,...,...,...,...
386929,2019-10-01 16:56:05,view,1801592,2053013554415534427,electronics.video.tv,sony,514.250000,517529590,6e659e14-addc-495b-9bdf-19a113a7905d,1,1
386930,2019-10-01 16:56:06,view,1004497,2053013555631882655,electronics.smartphone,nokia,159.330002,555610488,df50b14c-6581-4af0-957f-139d0f18a864,1,1
386931,2019-10-01 16:56:06,view,5100468,2053013553341792533,electronics.clocks,elari,47.770000,513024938,0908294b-85a1-4b82-8349-cf434e50e0b2,1,1
386932,2019-10-01 16:56:06,view,1801739,2053013554415534427,electronics.video.tv,samsung,282.200012,547896076,022c5a7b-2fa4-4cde-9913-6671c4de2626,1,1


In [17]:
model = initialize_model(X_processed, y_processed)

✅ Model initialized


In [18]:
train_model(model=model, X=X_train, y=y_train)

✅ Model trained


,boosting_type,'gbdt'
,num_leaves,40
,max_depth,5
,learning_rate,0.04
,n_estimators,200
,subsample_for_bin,200000
,objective,'binary'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [19]:
base_spw(y_train)

39.99466025897744

In [20]:
import joblib
import pickle
import os

In [21]:
# path = os.path.join('convertiq', 'raw_data')
joblib.dump(model, 'model_lgbm_baseline.pkl')

['model_lgbm_baseline.pkl']

In [22]:
X_test.head()

,total_events,total_views,total_carts,total_purchases,n_sessions,n_days_active,has_ever_carted,has_ever_purchased,view_to_cart_ratio,cart_to_purchase_ratio,avg_session_duration,median_session_duration,max_session_duration,avg_events_per_session,max_events_per_session
267494,2,2,0,0,1,1,0,0,0.000,0.0,13.0,13.0,13.0,2.0,2
366659,1,1,0,0,1,1,0,0,0.000,0.0,0.0,0.0,0.0,1.0,1
203354,3,3,0,0,3,1,0,0,0.000,0.0,0.0,0.0,0.0,1.0,1
67893,2,2,0,0,1,1,0,0,0.000,0.0,19.0,19.0,19.0,2.0,2
187652,19,16,2,1,1,1,1,1,0.125,0.5,647.0,647.0,647.0,19.0,19


In [23]:
X_test.shape

(74901, 15)

In [25]:
X_test.index.nunique()

74901

In [26]:
y_test.shape

(74901,)

In [27]:
y_test.head()

267494    0
366659    0
203354    0
67893     0
187652    0
Name: label, dtype: int64

In [29]:
(y_test == 1).sum()

np.int64(1895)